# 41 Fast Pulse Bandwidth

How the useful dressed-state regime becomes a device-level bandwidth requirement.

**Builds on:** `37_dressed_state_optimization.ipynb`

## Central question

> What pulse bandwidth supports the useful dressed-state regime for noise suppression and cavity-compatible control?

## Working statement

Useful dressed-state operation requires fast pulses, and fast pulses shift the dominant device constraint toward bandwidth, IDT geometry, and piezoelectric efficiency.

The seminar outlook slide provides the device anchor:

\[
t_{IDT} \sim 10\,\mathrm{ns}
\]

The device supports fast-pulse operation when its impulse response is shorter than the pulse timescale:

\[
t_{IDT} < t_{pulse}
\]


## 1. Setup

Notebook 37 identified a useful dressed-state regime.

Notebook 41 asks whether the device can access and use that regime through fast pulses.

The modeling sequence is:

\[
\text{useful dressed-state regime}
\rightarrow
\text{fast pulses}
\rightarrow
\text{bandwidth}
\rightarrow
\text{IDT geometry}
\rightarrow
\text{piezoelectric efficiency}
\]


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120


## 2. Pulse duration and bandwidth

A short pulse requires broad bandwidth.

Using the simple relation:

\[
B_{required} \sim \frac{1}{t_{pulse}}
\]

where `B_required` is a characteristic bandwidth and `t_pulse` is the pulse duration.

The seminar slide anchors the current device response near:

\[
t_{IDT} \sim 10\,\mathrm{ns}
\]

which corresponds to a characteristic bandwidth near:

\[
B_{IDT} \sim \frac{1}{t_{IDT}}
\]


In [ ]:
# Pulse durations in nanoseconds.
t_pulse_ns = np.linspace(1, 50, 500)

# Required bandwidth in MHz using B ~ 1/t.
# 1/ns = 1000 MHz.
B_required_MHz = 1000 / t_pulse_ns

# Seminar anchor.
t_IDT_ns = 10
B_IDT_MHz = 1000 / t_IDT_ns

t_IDT_ns, B_IDT_MHz


## 3. Pulse bandwidth requirement

The device response time sets a practical line on the pulse-duration axis.

Pulses longer than the IDT response are easier to support.

Pulses shorter than the IDT response motivate bandwidth engineering.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(t_pulse_ns, B_required_MHz, linewidth=2.5, label="Required bandwidth")
ax.axvline(t_IDT_ns, linestyle="--", linewidth=1.5, label=r"$t_{IDT} \sim 10$ ns")
ax.axhline(B_IDT_MHz, linestyle=":", linewidth=1.5, label=r"$B_{IDT} \sim 100$ MHz")

ax.fill_between(
    t_pulse_ns,
    B_required_MHz,
    B_required_MHz.max(),
    where=t_pulse_ns < t_IDT_ns,
    alpha=0.12,
)

ax.text(3.0, 650, "bandwidth\nengineering\nregime", ha="center", va="center", fontsize=10)
ax.text(28, 180, "IDT-supported\npulse regime", ha="center", va="center", fontsize=10)

ax.set_title("Pulse Duration Sets the Bandwidth Requirement")
ax.set_xlabel("Pulse duration, ns")
ax.set_ylabel("Required bandwidth, MHz")
ax.set_ylim(0, 1050)
ax.grid(True, alpha=0.3)
ax.legend()

output = FIGURES / "41_pulse_bandwidth_requirement.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 4. IDT finger-count tradeoff

Cornell's outlook slide points to a design tradeoff:

- fewer IDT fingers increase bandwidth
- fewer IDT fingers reduce conversion efficiency
- improved piezoelectric efficiency can recover drive strength

This notebook models that tradeoff schematically.


In [ ]:
# Normalized IDT finger count.
N = np.linspace(0.2, 1.0, 400)

# Schematic scaling:
# More fingers -> more efficiency.
# More fingers -> narrower bandwidth.
bandwidth_norm = 1 / N
efficiency_norm = N

# Normalize bandwidth to max 1 for display.
bandwidth_norm = bandwidth_norm / bandwidth_norm.max()

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(N, bandwidth_norm, linewidth=2.5, label="Bandwidth")
ax.plot(N, efficiency_norm, linewidth=2.5, label="Conversion efficiency")

ax.set_title("IDT Finger Count Shifts Bandwidth and Efficiency")
ax.set_xlabel("Normalized IDT finger count")
ax.set_ylabel("Normalized device response")
ax.grid(True, alpha=0.3)
ax.legend()

output = FIGURES / "41_idt_finger_tradeoff.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 5. Efficiency recovery

The outlook slides suggest a path forward:

\[
\text{reduce IDT fingers}
\rightarrow
\text{increase bandwidth}
\rightarrow
\text{recover drive strength through higher piezoelectric efficiency}
\]

Lithium niobate on diamond was presented as one route toward higher piezoelectric efficiency.


In [ ]:
platforms = pd.DataFrame(
    [
        {
            "Platform": "AlN on diamond",
            "Relative piezoelectric efficiency": 1.0,
            "Interpretation": "reference device platform",
        },
        {
            "Platform": "LiNbO3 on diamond",
            "Relative piezoelectric efficiency": 3.0,
            "Interpretation": "seminar outlook: larger effective bandwidth / stronger drive opportunity",
        },
    ]
)

platforms


## 6. Bandwidth / efficiency design map

The device target combines two requirements:

1. high enough bandwidth for fast pulses
2. high enough piezoelectric efficiency for strong mechanical drive

The favorable region sits where both are available together.


In [ ]:
finger_count = np.linspace(0.2, 1.0, 300)
piezo_eff = np.linspace(0.5, 3.5, 300)

F, E = np.meshgrid(finger_count, piezo_eff)

# Schematic responses.
B = 1 / F
B = B / B.max()

drive = E * F
drive = drive / drive.max()

# Design score rewards bandwidth and drive together.
score = B * drive

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(
    score,
    origin="lower",
    aspect="auto",
    extent=[finger_count.min(), finger_count.max(), piezo_eff.min(), piezo_eff.max()],
)

contours = ax.contour(
    F,
    E,
    score,
    levels=[0.25, 0.45, 0.65, 0.85],
    colors="black",
    linewidths=0.8,
)
ax.clabel(contours, inline=True, fontsize=8)

ax.scatter([0.8], [1.0], s=80, label="reference platform")
ax.scatter([0.4], [3.0], s=80, label="high-efficiency fast-pulse region")

ax.text(0.34, 3.18, "fast-pulse-ready\nregion", fontsize=10, ha="center")
ax.text(0.78, 1.2, "reference\nregion", fontsize=10, ha="center")

ax.set_title("Bandwidth / Efficiency Design Map")
ax.set_xlabel("Normalized IDT finger count")
ax.set_ylabel("Relative piezoelectric efficiency")
ax.legend(loc="upper right")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Device readiness score")

output = FIGURES / "41_bandwidth_efficiency_map.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 7. Pulse-support table

The `t_IDT ~ 10 ns` anchor makes it possible to classify candidate pulse durations.


In [ ]:
candidate_pulses = np.array([2, 5, 10, 20, 50])

pulse_table = pd.DataFrame(
    {
        "pulse_duration_ns": candidate_pulses,
        "required_bandwidth_MHz": 1000 / candidate_pulses,
        "IDT_anchor_ns": t_IDT_ns,
        "device_regime": np.where(
            candidate_pulses > t_IDT_ns,
            "IDT-supported pulse regime",
            np.where(
                candidate_pulses == t_IDT_ns,
                "transition regime",
                "bandwidth engineering regime",
            ),
        ),
    }
)

pulse_table


## 8. Interpretation

Notebook 37 identified the useful dressed-state regime.

Notebook 41 shows how that regime becomes an engineering requirement:

\[
\text{dressed-state operation}
\rightarrow
\text{fast pulses}
\rightarrow
\text{bandwidth}
\rightarrow
\text{IDT geometry}
\rightarrow
\text{piezoelectric efficiency}
\]

The seminar outlook points to a combined design target:

> high bandwidth and high piezoelectric efficiency together.

This makes lithium-niobate-on-diamond and related high-efficiency platforms natural next-step candidates for fast-pulse noise suppression and cavity-compatible operation.


## 9. Next notebook

Notebook 47 can focus on the material/device platform.

Candidate title:

> `47_piezoelectric_efficiency.ipynb`

Candidate question:

> How does higher piezoelectric efficiency shift the available bandwidth and drive-strength design space?

Candidate outputs:

- platform comparison
- efficiency multiplier model
- IDT geometry map
- cavity-compatible device score
